In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import math
import random

# Training result

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, NullLocator


def scientific_tick(value, position):
    """
    Format logarithmic ticks as:
    1E-02, 1E-01, 1E+00, 1E+01, ...
    """
    if value <= 0:
        return ""

    exponent = int(np.round(np.log10(value)))
    return f"1E{exponent:+03d}"


def plot_training_validation(
    classical_training,
    classical_validation,
    krylov_training,
    krylov_validation,
    hslm_training,
    hslm_validation,
    sigma_squared=0.65,
    save_name="training_validation_comparison.pdf",
):
    """
    Create a 2-by-3 figure:

        Classical LM       Krylov Subspace LM       HSLM
        Training           Training                  Training
        Validation         Validation                Validation

    Each input should be a list containing one array for each initial guess.
    """

    training_data = [
        classical_training,
        krylov_training,
        hslm_training,
    ]

    validation_data = [
        classical_validation,
        krylov_validation,
        hslm_validation,
    ]

    method_titles = [
        "Classical LM",
        "Krylov Subspace LM",
        "HSLM",
    ]

    panel_labels = [
        ["A1", "B1", "C1"],
        ["A2", "B2", "C2"],
    ]

    fig, axes = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(15, 6.4),
        sharex=False,
        sharey=False,
    )

    # ---------------------------------------------------------
    # Main formatting parameters
    # ---------------------------------------------------------
    tick_fontsize = 13
    axis_label_fontsize = 14
    title_fontsize = 19
    panel_fontsize = 17.5
    legend_fontsize = 9.4
    line_width = 1.8
    spine_width = 1.3
    tick_width = 1.4
    tick_length = 5

    for column in range(3):

        # =====================================================
        # Top row: training error
        # =====================================================
        ax_train = axes[0, column]

        for run_number, error_values in enumerate(training_data[column], start=1):
            error_values = np.asarray(error_values, dtype=float)
            iterations = np.arange(len(error_values))

            ax_train.plot(
                iterations,
                error_values,
                linewidth=line_width,
                label=f"Initial Guess {run_number}",
            )

        ax_train.axhline(
            y=sigma_squared,
            linestyle="--",
            linewidth=1.7,
            color="gray",
            label=r"$\sigma^2$",
        )

        ax_train.set_title(
            method_titles[column],
            fontsize=title_fontsize,
            fontweight="bold",
            pad=31,
        )

        # =====================================================
        # Bottom row: validation error
        # =====================================================
        ax_validation = axes[1, column]

        for run_number, error_values in enumerate(
            validation_data[column], start=1
        ):
            error_values = np.asarray(error_values, dtype=float)
            iterations = np.arange(len(error_values))

            ax_validation.plot(
                iterations,
                error_values,
                linewidth=line_width,
            )

        # =====================================================
        # Format both rows
        # =====================================================
        for row, ax in enumerate([ax_train, ax_validation]):

            ax.set_yscale("log")
            ax.set_xlim(-0.4, 35.4)
            ax.set_xticks(np.arange(0, 36, 5))

            if row == 0:
                ax.set_ylim(1e-1, 1.5e3)
                ax.set_yticks([1e-1, 1e0, 1e1, 1e2, 1e3])
            else:
                ax.set_ylim(7e-3, 1.5e3)
                ax.set_yticks([1e-2, 1e-1, 1e0, 1e1, 1e2, 1e3])

            ax.yaxis.set_major_formatter(
                FuncFormatter(scientific_tick)
            )

            # Remove all minor ticks
            ax.xaxis.set_minor_locator(NullLocator())
            ax.yaxis.set_minor_locator(NullLocator())

            # Only bottom x ticks and left y ticks
            ax.tick_params(
                axis="x",
                which="major",
                bottom=True,
                top=False,
                labelbottom=True,
                direction="out",
                length=tick_length,
                width=tick_width,
                labelsize=tick_fontsize,
                pad=3,
            )

            ax.tick_params(
                axis="y",
                which="major",
                left=True,
                right=False,
                labelleft=True,
                direction="out",
                length=tick_length,
                width=tick_width,
                labelsize=tick_fontsize,
                pad=3,
            )

            # Make all tick numbers bold
            for tick_label in ax.get_xticklabels():
                tick_label.set_fontweight("bold")

            for tick_label in ax.get_yticklabels():
                tick_label.set_fontweight("bold")

            # Axis-border formatting
            for spine in ax.spines.values():
                spine.set_linewidth(spine_width)
                spine.set_color("0.25")

            # Panel labels: A1, A2, B1, ...
            ax.text(
                -0.01,
                1.045,
                panel_labels[row][column],
                transform=ax.transAxes,
                fontsize=panel_fontsize,
                fontweight="bold",
                horizontalalignment="left",
                verticalalignment="bottom",
            )

    # ---------------------------------------------------------
    # Axis labels
    # ---------------------------------------------------------
    axes[0, 0].set_ylabel(
        "Training Error",
        fontsize=axis_label_fontsize,
        fontweight="bold",
    )

    axes[1, 0].set_ylabel(
        "Validation Error",
        fontsize=axis_label_fontsize,
        fontweight="bold",
    )

    for column in range(3):
        axes[1, column].set_xlabel(
            "Iteration",
            fontsize=axis_label_fontsize,
            fontweight="bold",
        )

    # Legend only in the first panel
    axes[0, 0].legend(
        loc="upper right",
        fontsize=legend_fontsize,
        frameon=False,
        handlelength=2.2,
    )

    plt.subplots_adjust(
        left=0.075,
        right=0.985,
        bottom=0.12,
        top=0.86,
        wspace=0.24,
        hspace=0.32,
    )

    if save_name is not None:
        plt.savefig(
            save_name,
            dpi=400,
            bbox_inches="tight",
        )

    plt.show()

# ============================================================
# Example data
# Replace these arrays with your actual training-error values
# ============================================================

classical_errors = [
     [599.3325241199411,179.9169305233625, 179.9169305233625, 179.9169305233625, 56.56567042494202,
        56.56567042494202, 56.56567042494202, 27.589877141381532, 27.589877141381532,
        22.27192562152479, 15.226122952828185, 15.226122952828185, 10.509918773609975,
        5.5058214030085075, 3.9927675969999306, 3.4873131639458093, 3.4873131639458093,
        2.1631182663935293, 2.035680585549925, 2.035680585549925, 1.6173549529159719,
        1.170050229781236, 1.0318187294952135, 1.0318187294952135, 0.8585335180512788,
        0.8270795157452591, 0.8270795157452591, 0.7778783288404852, 0.7565190785831503,
        0.7546210501054579, 0.7546210501054579, 0.7111321880703546, 0.7000450632803519,
        0.6886620359001677, 0.6886620359001677, 0.6744365407114552],

    [606.7482737017061, 606.7482737017061, 305.36967881021087, 305.36967881021087, 305.36967881021087,
        279.96248647631955, 140.02251983423204, 140.02251983423204, 42.13458749589885,
        42.13458749589885, 40.29593285381301, 20.2967780097667, 20.2967780097667,
        8.136357303377121, 8.136357303377121, 3.733519900393655, 2.7900262147709283,
        2.7900262147709283, 2.1678253783016634, 2.0143295766702964, 1.8516080912116093,
        1.8516080912116093, 1.2532889724247605, 1.1868348196489096, 1.079518823522675,
        0.9523227036699323, 0.9523227036699323, 0.8319971726689358, 0.8072555351475048,
        0.7806195630879308, 0.7806195630879308, 0.7311365881795702, 0.7311365881795702,
        0.7123269931276294, 0.7056993447078681, 0.6983461311873593],

    [598.0743021761541,317.3889453744474, 317.3889453744474, 317.3889453744474, 312.89049220546644,
        158.05422644931312, 158.05422644931312, 57.45289592546596, 57.45289592546596,
        45.53596315020143, 45.53596315020143, 33.53901469592939, 28.781631053567864,
        27.43031116228894, 23.937461522161023, 23.937461522161023, 22.15053062021939,
        22.15053062021939, 11.036649312251816, 3.800794018181059, 3.800794018181059,
        3.069944070820866, 1.8263058214024677, 1.8263058214024677, 1.8263058214024677,
        1.8263058214024677, 1.5067373522971643, 1.2048090929849946, 1.2048090929849946,
        1.1447793932479329, 1.0718275223070366, 1.0460688124986117, 1.0460688124986117,
        1.0154532917081855, 0.9922538395029238, 0.9790176635944785],

     [666.487176186392, 666.487176186392, 330.4164021610494, 207.65768521879124, 93.41902526839897,
        70.52401720097147, 70.52401720097147, 70.52401720097147, 37.392135481995645,
        37.392135481995645, 21.48803340767973, 21.48803340767973, 10.231552367125854,
        9.559417063033143, 9.559417063033143, 9.559417063033143, 4.869328896948646,
        3.5494919510018983, 3.1899159634708, 2.5438724311239826, 2.5438724311239826,
        1.8134987161166518, 1.5111037890676433, 1.5111037890676433, 1.1897151585244767,
        1.1302516066875732, 1.1302516066875732, 0.9942189072443492, 0.9105549486217417,
        0.9105549486217417, 0.8434854463696869, 0.7963172315863399, 0.7703508489074036,
        0.7332558797659176, 0.7332558797659176, 0.686149613495817],
 [593.6090942143439, 243.9577610881797, 243.9577610881797, 160.32001321758415, 160.32001321758415,
        70.24521680135626, 70.24521680135626, 26.637396562238212, 26.637396562238212,
        26.637396562238212, 15.69380824522123, 9.35438297458985, 9.23184887564184,
        9.23184887564184, 3.704156243285358, 3.3275466373786666, 3.267318440884676,
        3.267318440884676, 1.1720165808635512, 0.9432967709582066, 0.9432967709582066,
        0.78822727709097, 0.7717544663069579, 0.7717544663069579, 0.7088013614837475,
        0.7074799869260009, 0.6993882730163689, 0.6884784401639731, 0.6884784401639731,
        0.6541273494148256, 0.6541273494148256, 0.6430208947045885, 0.6409171243629821,
      0.6409171243629821,0.6409171243629821,0.6409171243629821]
]

krylov_errors = [
    [599.3325241199411,179.91704743794884, 179.91704743794884, 179.91704743794884,
        56.569369637501126, 56.569369637501126, 56.569369637501126,
        27.58983842999727, 27.58983842999727, 22.271504126172587,
        15.228805198643627, 15.228805198643627, 10.511161633162805,
        5.506124966181083, 3.9936943811003482, 3.4860189333300005,
        3.4860189333300005, 2.1635446619509016, 2.035558922418463,
        2.035558922418463, 1.616848990797768, 1.1696925780203054,
        1.0295033903341269, 1.0295033903341269, 0.8585679366524981,
        0.8267535022974649, 0.8267535022974649, 0.777833302670906,
        0.7566522826547978, 0.7554342234358261, 0.7551253507263528,
        0.6953415172777293, 0.6953415172777293, 0.6953415172777293,
        0.6668054958437472, 0.6663962507182201],
 [606.7482737017061,606.7482737017061, 305.36967881281674, 305.36967881281674,
        305.36967881281674, 279.9624864878902, 140.02251985837725,
        140.02251985837725, 42.13458746930504, 42.13458746930504,
        40.2959328235493, 20.296778038279175, 20.296778038279175,
        8.136357599656593, 8.136357599656593, 3.733519856341256,
        2.790026187178165, 2.790026187178165, 2.167825387654171,
        2.014329538465992, 1.8516064135972687, 1.8516064135972687,
        1.2532889896199528, 1.1868347501036733, 1.0795053960072916,
        0.9523431106464774, 0.9523431106464774, 0.831998652960618,
        0.8072837303906272, 0.7821010235035953, 0.7821010235035953,
        0.7312086757656416, 0.7312086757656416, 0.7123435091805909,
        0.7057199365837776, 0.6984258845871514],
 [598.0743021761541,317.3896776995741, 317.3896776995741, 317.3896776995741,
        312.89063761350457, 158.0687582363011, 158.0687582363011,
        57.455252579672845, 57.455252579672845, 45.47940737356403,
        45.47940737356403, 33.53335949523101, 28.764545561982732,
        27.472373600348448, 25.975185773350592, 25.975185773350592,
        25.975185773350592, 15.398990849033021, 5.9044672871187895,
        2.9688610477944026, 2.9688610477944026, 2.521538802373974,
        1.9131481623615654, 1.9131481623615654, 1.9131481623615654,
        1.9131481623615654, 1.4471457231332092, 1.1539931920222997,
        1.1174272433402803, 1.0935645972652637, 1.0935645972652637,
        1.0561516408468266, 0.9854076617573381, 0.9854076617573381,
        0.9488079427914549, 0.9190883772816218],
 [666.487176186392,666.487176186392, 330.4164021616961, 207.65787594927633,
        93.7286330474138, 67.19477770495831, 67.19477770495831,
        67.19477770495831, 34.69334665398965, 34.69334665398965,
        17.967221286041568, 12.017187819489603, 12.017187819489603,
        10.200953066288292, 10.200953066288292, 5.774584639845475,
        4.982610262446936, 4.982610262446936, 2.899193151905413,
        2.851462951631142, 2.851462951631142, 1.5752080141936808,
        1.4390848995470746, 1.4390848995470746, 1.1031969064847198,
        1.0227919651953932, 0.9838119658908978, 0.9175589195790155,
        0.9175589195790155, 0.7161563858960102, 0.6964169465694792,
        0.6964169465694792, 0.6535162899232401, 0.649524955832508
       , 0.649524955832508, 0.649524955832508],
[593.6090942143439,243.9562656071609, 243.9562656071609, 160.35699049280672,
        160.35699049280672, 65.75094956267415, 65.75094956267415,
        25.59735053754988, 25.59735053754988, 25.59735053754988,
        14.151700800872122, 10.525989558182912, 6.727159791100188,
        6.727159791100188, 4.077858429333875, 2.2481183947906107,
        2.2481183947906107, 1.2459213976372627, 0.9843359679275987,
        0.9843359679275987, 0.8633170498020268, 0.818673022723959,
        0.7695729764167601, 0.7695729764167601, 0.7288695650577197,
        0.7194707934981991, 0.7194707934981991, 0.6922684404344358,
        0.6827283093119577, 0.6787290161688682, 0.6787290161688682,
        0.6610258095581599, 0.6572725409724554, 0.6572725409724554
       , 0.6572725409724554, 0.6572725409724554]
]

hslm_errors = [
     [599.3325241199411,535.3926336851425, 442.96918826951145, 341.5157263251788, 268.7780097884,
        225.30533744295883, 222.08878150172663, 213.2881711278636, 129.1979859968923,
        119.905242693991, 54.476278481821154, 29.176972033159075, 12.618715710068178,
        5.7698608122724675, 4.488724764798779, 3.762215248051037, 3.119222787392049,
        2.5453094161738394, 2.1660834838295386, 1.9044550041284514, 1.6159406890103913,
        1.5665495377563365, 1.3769746659362383, 1.3241296471447452, 1.1717739488265044,
        1.1429838113350141, 1.1419394975738166, 0.9163494999441311, 0.8478174626154512,
        0.7819199105595924, 0.7361908923374998, 0.7154284155859353, 0.7022067714630881,
        0.6808592607688475, 0.6669873824362854, 0.660703330060815]
, [606.7482737017061,541.8252140123246, 447.6654472389141, 343.64259258283516, 262.2737525605216,
        184.39498171766846, 163.32940870903414, 152.8004020371148, 151.58748528910183,
        64.17800462629997, 52.13785338863146, 43.78627953831563, 25.1180796842815,
        18.352240062384958, 9.824783785833175, 7.938343829513251, 6.839399799062989,
        6.403749508272502, 3.3121981900105384, 2.693484891735317, 2.29267159601697,
        1.7781056509873192, 1.562162204117883, 1.2832362554582912, 1.1564196560291908,
        1.0540759616502928, 1.0113505216098988, 0.9160265345413391, 0.883490795935046,
        0.8300482320624837, 0.8186632397853053, 0.7737022085721754, 0.7557075097427002,
        0.7354479276237226, 0.7134715204869361, 0.7078025932797558]
, [598.0743021761541, 534.5875690071064, 442.52666854152835, 340.8212275162923, 267.4940570620111,
        193.5943444560718, 182.65936034934825, 165.54671132858965, 114.65800000905564,
        100.07795260082408, 80.0357082717229, 55.56802947571785, 19.842713716172888,
        15.490502973902567, 11.027389214540465, 9.024824061262589, 6.520990453957544,
        4.383034001753878, 3.3211906538982743, 2.528612903117307, 1.39078045303358,
        1.1023624257215405, 0.9262145180207776, 0.8055290265990759, 0.7794064492432331,
        0.7414373750943456, 0.7214863039242017, 0.701852299461879, 0.6924748077076006,
        0.6839953718031877, 0.6735575186954, 0.6690422828289533, 0.6690422828289533,
        0.6690422828289533, 0.6690422828289533, 0.6690422828289533]
,[666.487176186392,591.2119081457678, 481.92032868505083, 361.92775232832247, 276.94799682961497,
        227.6499400449304, 223.30488120118454, 211.20850045341234, 192.80854202052907,
        163.23713777010593, 67.19348549088852, 56.262696779780704, 38.20515930419215,
        19.994001418669253, 4.033187803931911, 2.956639205395423, 2.3494990706746717,
        1.749986602068173, 1.3876547245886866, 1.1462735874259127, 0.9298372480075741,
        0.8717498113350138, 0.7820785706097043, 0.7588572396013035, 0.7386080010383109,
        0.7161770436606586, 0.7060342864458077, 0.6946437427477223, 0.6816643238817629,
        0.6706965211533177, 0.6646289287849318, 0.6539489925019077, 0.6524620605228237
       , 0.6524620605228237, 0.6524620605228237, 0.6524620605228237]
, [593.6090942143439,530.9891756209419, 440.09321426545057, 339.39317289977316, 267.3901522720732,
        225.1518678535262, 217.68141679911182, 217.68141679911182, 208.1781640982125,
        180.41071862364075, 169.508624308332, 145.17741594866598, 59.054906230895035,
        49.69896080184875, 48.03243715234465, 17.73808915601245, 15.487838230199035,
        10.356662367408905, 10.205650543370261, 6.415368018130068, 5.17537883342172,
        4.091306122750394, 3.008297251109856, 2.737212573132232, 2.1359834832218128,
        1.5776014890743968, 1.274929265065109, 1.1427725561092321, 0.9850904269410555,
        0.8726462055240032, 0.8172115248796865, 0.7167757466883317, 0.6955539455201643,
        0.6681727121199383, 0.6565315263057896, 0.6545424642654005]
]

classical_validation_errors = [[638.4397171122417, 181.5943042760947, 181.5943042760947, 181.5943042760947, 58.6287624226155,
        58.6287624226155, 58.6287624226155, 27.75500446846024, 27.75500446846024,
        22.041828129613997, 14.864833736289498, 14.864833736289498, 9.767562403099387,
        4.623728639391928, 3.246010835485057, 2.75086677967369, 2.75086677967369,
        1.490201386854208, 1.455377501323135, 1.455377501323135, 1.049156029624009,
        0.5830852942269581, 0.44131138374230444, 0.44131138374230444, 0.2637350661202978,
        0.2322213775418628, 0.2322213775418628, 0.1821192788081464, 0.1578758027864998,
        0.1519959322713505, 0.1519959322713505, 0.11722173034805103, 0.10483018895464696,
        0.09166907745377403, 0.09166907745377403, 0.08019800084139596],
                               [642.3595592715828,642.3595592715828, 305.9872662853282, 305.9872662853282, 305.9872662853282,
        285.23354146223517, 148.86163490422504, 148.86163490422504, 41.67313063357112,
        41.67313063357112, 41.04019166099535, 20.20629641248705, 20.20629641248705,
        7.186136842751509, 7.186136842751509, 2.9265394226731742, 2.0975992175041465,
        2.0975992175041465, 1.4355096399305378, 1.2817423090588769, 1.1408082880356343,
        1.1408082880356343, 0.5665271198743705, 0.5007614527161501, 0.417561667757206,
        0.32654532594873253, 0.32654532594873253, 0.20418279327547378, 0.1894728910811691,
        0.17396569072746712, 0.17396569072746712, 0.12452285206337502, 0.12452285206337502,
        0.10538136736514973, 0.1012686244152547, 0.09615895196354199],
                               [634.7882883798769,328.7137367794112, 328.7137367794112, 328.7137367794112, 321.8714619845155,
        163.3370861892402, 163.3370861892402, 61.03676330598184, 61.03676330598184,
        46.12359460715641, 46.12359460715641, 30.943730992805236, 28.378784622596545,
        24.95257324287932, 23.793690524780484, 23.793690524780484, 22.849024419180793,
        22.849024419180793, 10.908584332971346, 2.969579370345341, 2.969579370345341,
        2.5495913481185437, 1.0807755434408706, 1.0807755434408706, 1.0807755434408706,
        1.0807755434408706, 1.0092556093998037, 0.6276123535334628, 0.6276123535334628,
        0.5977745288268889, 0.49714930289667164, 0.4821379568218333, 0.4821379568218333,
        0.43507409995508534, 0.42021370873367947, 0.40000292446211194],
                               [707.1423725721296, 707.1423725721296, 332.51474939804757, 213.40603313101747, 92.91614722521379,
        71.47789246807105, 71.47789246807105, 71.47789246807105, 37.005773752893774,
        37.005773752893774, 21.070557957624626, 21.070557957624626, 9.549189990802715,
        8.98516167653794, 8.98516167653794, 8.98516167653794, 4.312557040765845,
        2.9622306835484475, 2.52299483251297, 1.9768850638895417, 1.9768850638895417,
        1.2193291636136834, 0.8634546756026205, 0.8634546756026205, 0.5604043399716885,
        0.4890538570732366, 0.4890538570732366, 0.37630487679905095, 0.3072641224142619,
        0.3072641224142619, 0.2471410796728631, 0.2082367536629741, 0.17937238692676669,
        0.14052201980074386, 0.14052201980074386, 0.09893712657624643],
                               [627.4277869540301,244.27742458134966, 244.27742458134966, 161.16726226914537, 161.16726226914537,
        69.6530993867735, 69.6530993867735, 24.58542998350093, 24.58542998350093,
        24.58542998350093, 14.307982233206726, 8.041621781617161, 8.580218823582916,
        8.580218823582916, 2.9873675308129606, 2.7510716821303993, 2.6076014660070297,
        2.6076014660070297, 0.6369365453048748, 0.3889208817238044, 0.3889208817238044,
        0.21901900210484798, 0.199511112748237, 0.199511112748237, 0.12841562989754396,
        0.1245871542002792, 0.11143666394729786, 0.10404119766255522, 0.10404119766255522,
        0.06806261668171305, 0.06806261668171305, 0.057980099221697265, 0.05501479249788003, 
                                0.05501479249788003, 0.05501479249788003, 0.05501479249788003]]
krylov_validation_errors = [[638.4397171122417,181.59491025363673, 181.59491025363673, 181.59491025363673, 58.631727987374575,
        58.631727987374575, 58.631727987374575, 27.75545547327993, 27.75545547327993,
        22.041423657247904, 14.8678786397564, 14.8678786397564, 9.768728629671116,
        4.624134064893773, 3.246884155800328, 2.7494429152652557, 2.7494429152652557,
        1.4904405900275577, 1.4551024641479728, 1.4551024641479728, 1.0485814288581217,
        0.5826695149410624, 0.43921097273761417, 0.43921097273761417, 0.26391359696237715,
        0.2319741636148432, 0.2319741636148432, 0.18205421933593727, 0.1579197060611054,
        0.15251632381860122, 0.15131745508060318, 0.09635591649416939, 0.09635591649416939,
        0.09635591649416939, 0.07110873674675516, 0.070671137656028],
                            [642.3595592715828,642.3595592715828, 305.98726628763274, 305.98726628763274,
        305.98726628763274, 285.2335414769352, 148.86163493494038,
        148.86163493494038, 41.673130612807626, 41.673130612807626,
        41.040191622555405, 20.20629650332204, 20.20629650332204,
        7.186137158297069, 7.186137158297069, 2.9265393858218833,
        2.097599170789521, 2.097599170789521, 1.435509647502268,
        1.281742276402058, 1.1408067427346313, 1.1408067427346313,
        0.5665271362005218, 0.5007613690748799, 0.4175481122209047,
        0.3265726489921918, 0.3265726489921918, 0.2041770928892946,
        0.18950405783261243, 0.17557731683622432, 0.17557731683622432,
        0.12462487632239719, 0.12462487632239719, 0.10542907328065004,
        0.10130532510682017, 0.09623645595],
                            [634.7882883798769,328.71455149443284, 328.71455149443284, 328.71455149443284,
        321.87365870746993, 163.347353133426, 163.347353133426,
        61.0352532857609, 61.0352532857609, 46.06742100106501,
        46.06742100106501, 30.942796274195427, 28.363124868934968,
        25.014378224559614, 25.942356782010226, 25.942356782010226,
        25.942356782010226, 15.64654155496076, 5.603089564391031,
        2.3567635030098253, 2.3567635030098253, 2.039293351848224,
        1.2018972790518818, 1.2018972790518818, 1.2018972790518818,
        1.2018972790518818, 0.9496152818717559, 0.5848417289703283,
        0.5709114025345337, 0.5106800690671682, 0.5106800690671682,
        0.510204889631945, 0.40007884579737935, 0.40007884579737935,
        0.38224317250078377, 0.3325823905699174],
                            [707.1423725721296,707.1423725721296, 332.514749398558, 213.4074140446671,
        93.2741938930548, 68.55873077169099, 68.55873077169099,
        68.55873077169099, 33.960056531592485, 33.960056531592485,
        17.690649572855087, 11.901431832745992, 11.901431832745992,
        10.05499940095754, 10.05499940095754, 5.209409626848737,
        4.550594196060256, 4.550594196060256, 2.4052842233968423,
        2.205525164278224, 2.205525164278224, 0.9879951169041976,
        0.7801102470708077, 0.7801102470708077, 0.47462290700764287,
        0.38620894146389334, 0.36796889255075865, 0.31193996775594385,
        0.31193996775594385, 0.10951856324688122, 0.09754335821457223,
        0.09754335821457223, 0.05472504821263598, 0.053666698230817, 0.053666698230817, 0.053666698230817],
                            [627.4277869540301,244.27570923367088, 244.27570923367088, 161.21452362701424,
        161.21452362701424, 65.0782690347136, 65.0782690347136,
        23.845698089051574, 23.845698089051574, 23.845698089051574,
        12.666793290733958, 9.167992920547555, 6.013586443381923,
        6.013586443381923, 3.2670176398976976, 1.666396390391657,
        1.666396390391657, 0.629645594213227, 0.3866978711797822,
        0.3866978711797822, 0.27399340791154214, 0.23283233946356552,
        0.18753235488567102, 0.18753235488567102, 0.14764146731014602,
        0.13542854543559785, 0.13542854543559785, 0.11036733140123746,
        0.09943062165159518, 0.09199830498613132, 0.09199830498613132,
        0.07677601023694426, 0.07073972334230008, 0.07073972334230008
                            , 0.07073972334230008, 0.07073972334230008]]
hslm_validation_errors = [[638.4397171122417,569.0266085324797, 468.23842924214154, 357.32836045925245,
        275.90275907509056, 229.28408486851498, 223.92756211463913,
        214.38645097695164, 129.83022866524055, 120.45748948820008,
        54.217481064133054, 29.633906457150164, 12.723769155352231,
        5.340914591534933, 4.070683883833448, 3.297289191788872,
        2.6449797005558695, 2.0590167040545886, 1.6559981258651038,
        1.3645829203071926, 1.072088611797778, 1.0254890498843499,
        0.8245915881828735, 0.7802736154037093, 0.6275024450973824,
        0.594964877016666, 0.604693494584517, 0.35282515816160753,
        0.27107650264292055, 0.20361443388088865, 0.14866820042675044,
        0.12515929663440872, 0.109064080178718, 0.08456904818912808,
        0.06924069793831689, 0.06285664688142793],
                          [642.3595592715828,573.7401196943093, 473.45915438829223, 360.99581667864345,
        270.1186633106338, 187.6117406201242, 164.84490143825963,
        152.96398891199368, 154.9577006000008, 61.62724963641535,
        50.53969284166777, 42.28283513258256, 24.873140453711997,
        18.086148470510423, 9.105082770040216, 7.0703397398014145,
        6.048629568638885, 5.792352186232092, 2.5153367795004025,
        2.0152656225000753, 1.548307750154648, 1.1033657411519642,
        0.9327594405425745, 0.6294841689541706, 0.5318335999217295,
        0.4176611155660911, 0.38627878975187363, 0.29110466472016044,
        0.26339310904627716, 0.2139874429950716, 0.20432359073258022,
        0.16431003984753142, 0.14710758889575096, 0.13015990774721564,
        0.10885931508686456, 0.10469569893609716],
                          [634.7882883798769,567.1480883722871, 468.3995679279463, 357.6277231662348,
        274.9163246669058, 196.12314454574943, 185.60527591223337,
        168.62486636058264, 115.2750816176579, 100.95282453221554,
        80.45797788401129, 56.43583658472054, 18.409258955527285,
        14.569926405009435, 9.614501767727754, 8.050870455134314,
        5.67818565343562, 3.5261753569106205, 2.524757542360068,
        1.920969972026684, 0.7212084004414712, 0.48777723550971447,
        0.31918010798794255, 0.20452627134737378, 0.18300248271979774,
        0.14845880730490363, 0.12994994959943176, 0.11177280249561182,
        0.10392546902429149, 0.09410631878918377, 0.08706606393574876,
        0.08217467532121392,0.08217467532121392,0.08217467532121392,
                          0.08217467532121392,0.08217467532121392],
                          [707.1423725721296,627.6493448882335, 511.72523986960005, 382.8147350555989,
        288.2233401819719, 231.86360516978908, 227.6781366709746,
        214.7823671024326, 196.5515170564793, 167.48227444747292,
        67.14543604437195, 55.94171208262299, 37.13640778350607,
        18.992832761904072, 3.0579183801056615, 2.122739817535925,
        1.623622224150778, 1.0057449369761962, 0.7088215144305646,
        0.5564552881106458, 0.3209631308605617, 0.29488204337152363,
        0.19952432785674598, 0.17051584227392064, 0.15419133389677858,
        0.1253109556358433, 0.11804373757757143, 0.1017081058351962,
        0.09090915898469183, 0.07781518323702906, 0.07283378541886865,
        0.06212862179790594, 0.06095217102745039, 0.06095217102745039
                          , 0.06095217102745039, 0.06095217102745039],
                          [627.4277869540301,561.0591262081274, 464.1737645931234, 355.7748285403326,
        276.66895655604674, 229.202204989489, 221.79939505034864,
        221.79939505034864, 213.65402058811353, 183.2352729151622,
        177.74656784806066, 151.2411173810129, 57.038019841552654,
        47.774259877667205, 46.6633841064128, 15.90308399208241,
        14.02421229576669, 8.85418687961578, 9.020076723053874,
        5.219377475750193, 4.136129154485463, 3.032875111832921,
        2.094025449661287, 1.8748670871660107, 1.4227965811092027,
        0.8629208067805765, 0.5921042550622156, 0.4858220058033092,
        0.32683145452531365, 0.23012020438067549, 0.19905692972404074,
        0.10113107119975914, 0.08600781382043829, 0.062279721935724994,
        0.05242468190219094, 0.0512781750653019]]
plot_training_validation(
    classical_training=classical_errors,
    classical_validation=classical_validation_errors,
    krylov_training=krylov_errors,
    krylov_validation=krylov_validation_errors,
    hslm_training=hslm_errors,
    hslm_validation=hslm_validation_errors,
    sigma_squared=0.622,
    save_name="training_validation_comparison.svg",
)


# Adaptive subspace dimension selection

In [ ]:
t = np.arange(1, 36)
t1 = np.arange(1, 41)
arr1 = [102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102,102]
arr2 = [10,10,10,40,70,70,102,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40,40]

arr11 = [203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203]
arr22 = [20,20,20,20,20,80,80,80,80,80,80,80,20,80,20,80,80,80,20,80,20,80,80,80,20,80,20,80,20,80,20,80,20,80,20]

arr111 = [392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392,392]
arr222 = [39,39,39,39,39,156,156,156,156,156,39,156,156,156,39,156,39,156,156,39,156,156,39,156,39,156,39,156,156,39,156,39,156,39,156]

fig, ax = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(18, 8),
        sharex=False,
        sharey=False)

ax[0][0].scatter(t1,arr1,s=30)
ax[0][0].scatter(t1,arr2,s =30)
ax[0][0].set_ylim(0,110)
ax[0][0].set_yticks([10,40,70,102])
ax[0][0].set_yticklabels([10,40,70,102], size=12, fontweight="bold")
ax[0][0].set_xticklabels(ax[0][0].get_xticklabels(), size=12, fontweight="bold")
ax[0][0].set_title("Network 1", fontsize=18, fontweight="bold",pad=10)
ax[0][0].legend(["Krylov Subspace LM","HSLM"], fontsize=12)

ax[0][1].scatter(t,arr11,s=30)
ax[0][1].scatter(t,arr22,s =30)
ax[0][1].set_ylim(0,220)
ax[0][1].set_yticks([20,80,203])
ax[0][1].set_yticklabels([20,80,203], size=12, fontweight="bold")
ax[0][1].set_xticklabels(ax[0][1].get_xticklabels(), size=12, fontweight="bold")
ax[0][1].set_title("Network 2", fontsize=18, fontweight="bold",pad=10)

ax[0][2].scatter(t,arr111,s=30)
ax[0][2].scatter(t,arr222,s =30)
ax[0][2].set_ylim(0,440)
ax[0][2].set_yticks([39,156,392])
ax[0][2].set_yticklabels([39,156,392], size=12, fontweight="bold")
ax[0][2].set_xticklabels(ax[0][2].get_xticklabels(), size=12, fontweight="bold")
ax[0][2].set_title("Network 3", fontsize=18, fontweight="bold",pad=10)

ax[0][0].tick_params(length=6, width=3)
ax[0][1].tick_params(length=6, width=3)
ax[0][2].tick_params(length=6, width=3)
ax[0][0].set_ylabel("Dimension of Subspace", size=14, fontweight="bold")



acc1  = [
[ 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40],
[0.993501356483723,        0.9945943327085763,        0.9936578680352316,        0.9999988005260394,        0.9983039389674038,        0.9915861592825901,
0.9994647579651894,        0.9992119374138152,        0.999346536376986,        0.9999602069597394,        0.9999781131078748,        0.9999895216409185,
0.9998982674863155,        0.9999125538144856,        0.9999861122571068,        0.9999672592779841,        0.9999922646317074,        0.9999927352046115,
0.9998558971653578,        0.9999940472099198,        0.9999694796799684,        0.9999912901058722,        0.9999822077447942,        0.9999816744953669,
0.9999189483913467,        0.9999427994836614,        0.9999762718287171,        0.999967599710501,        0.99999512653739,        0.9999284616311461,
0.9999969541705366,        0.9999747611936359,        0.9999973353070052,        0.9999956170044265,        0.9999964754844454,        0.9999960504144161,
0.9999970945610782,        0.9999972093121132,        0.9999969109409043    ]
]
rej1 = [ [4, 5, 5, 6, 6, 7, 7, 7, 7, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37,
        38, 39, 40 ],
    [ 0.9896935716308831,        0.7878028116257141,        0.9894692863003762,        0.4379881045328762,        0.976675019863878,        0.18966162627829236,
        0.8363851746140488,        0.9523233930871355,        0.9875775451594674,        0.9879134545872823,        0.9294342421846323,        0.8494932031338164,
        0.845041873535939,        0.958774834998793,        0.956400574491104,        0.9660558413410646,        0.9489998572368639,        0.8981498939938503,
        0.8941801852892929,        0.8737626858260812,        0.7422014459680786,        0.9027921305916113,        0.6185760521543726,        0.9042014887021134,
        0.6564483198121244,        0.8713454337962591,        0.7154787663234652,        0.7975414141860028,        0.5988904981486031,        0.7924341017389249,
        0.6136964759279335,        0.7746246463888292,        0.7951226316964922,        0.7324744095322651,        0.8727215617164109,        0.6890243399393716,
        0.8803687269952667,        0.7179136571377616,        0.8405590781607051,        0.7560177198290761,        0.8666674014097515,        0.7833414378181264,
        0.870145289166047
    ]
]
acc2 = [
    [
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
        11, 12, 13, 14, 15, 16, 17, 18, 19, 20,
        21, 22, 23, 24, 25, 26, 27, 28, 29, 30,
        31, 32, 33, 34, 35
    ],
    [
        0.9985633692886337,
        0.99869777835602,
        0.998635147823505,
        0.9974862360710981,
        0.9931097549377924,
        0.9999697265603741,
        0.9998939106100917,
        0.9998555059801418,
        0.9999608373281185,
        0.9997031723406666,
        0.9999852489359099,
        0.9999342013473215,
        0.9923837097810303,
        0.9999974401121199,
        0.9939904941599462,
        0.9997329177922935,
        0.999988232187326,
        0.9999980806017479,
        0.992693601834359,
        0.9999961275455969,
        0.9927491554584876,
        0.9998566311098853,
        0.9999982323856292,
        0.9999989595484828,
        0.995941737688143,
        0.9998416588534678,
        0.9920542172321172,
        0.9986883234132601,
        0.9907670235928648,
        0.9999173738503105,
        0.9926414642948438,
        0.9993907131791621,
        0.992327244950028,
        0.9986815581757866,
        0.9914004798021016
    ]]
rej2 =[
    [
        6, 7, 8, 9, 10, 11, 12, 14, 16, 17,
        18, 20, 22, 23, 24, 26, 28, 30, 32, 34
    ],
    [
        0.9510493469423321,
        0.9353866191304014,
        0.9020176768636041,
        0.9805970093172001,
        0.9040370938446547,
        0.9879818631316972,
        0.8465629747872966,
        0.40214436482721966,
        0.0325477318395298,
        0.6743550050483806,
        0.9333219278014111,
        0.16488076158478038,
        0.005417799115615714,
        0.9857896546361791,
        0.9890458801116561,
        0.0023711975765285352,
        0.00011664062331251782,
        0.0033716730558678645,
        0.00038859495615210636,
        9.231930374719285e-05
    ]
]

acc3 =[
    [
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
        11, 12, 13, 14, 15, 16, 17, 18, 19, 20,
        21, 22, 23, 24, 25, 26, 27, 28, 29, 30,
        31, 32, 33, 34, 35
    ],
    [
        0.9998098228747535,
        0.9998497265795894,
        0.9998755962205627,
        0.9998506712900883,
        0.9980005705853938,
        0.9999994560250587,
        0.9999965998685493,
        0.9999965998685493,
        0.999991199612909,
        0.9999957818086087,
        0.9968923602332245,
        0.9999990314330176,
        0.9999970271233187,
        0.999998984105752,
        0.9956397358826001,
        0.999999840659044,
        0.9927025530254837,
        0.9999999368633841,
        0.9999999466321204,
        0.9985856152720661,
        0.9999999770035201,
        0.9999998834281519,
        0.9978354314082566,
        0.9999999896838636,
        0.9954018893876967,
        0.9999996127207494,
        0.9943359682477453,
        0.9999998394957561,
        0.9999999891469978,
        0.997046314356026,
        0.9999995815696127,
        0.9920529611293323,
        0.9999997698402648,
        0.9969810930152431,
        0.9999982948880863
    ]
]
rej3 =[
    [
        6, 7, 8, 9, 10, 12, 13, 14, 16, 18,
        19, 21, 22, 24, 26, 28, 29, 31, 33, 35
    ],
    [
        0.9881379514195194,
        0.9819895541885507,
        0.9819895541885507,
        0.9748520433846694,
        0.9881170708441452,
        0.9787282855314742,
        0.9727724591682203,
        0.9890169761996911,
        0.9678865848036021,
        0.9665205664604083,
        0.9890918435782274,
        0.9563578321771253,
        0.9891875086823578,
        0.7302815803231447,
        0.02840985123423328,
        0.031268082438917676,
        0.9869855417795237,
        0.15378096281575984,
        0.0354873797350113,
        0.005090245966812179
    ]
]


ax[1][0].scatter(acc1[0],acc1[1],s = 40 , marker = 'o', facecolors="none",    edgecolors="green")
ax[1][0].scatter(rej1[0],rej1[1],s =40, marker = 'x',   color='red')
ax[1][0].legend(["Accepted","Rejected"], fontsize=12, loc = "lower right")
ax[1][0].set_ylim(-0.05,1.05)
ax[1][0].set_yticklabels(ax[1][0].get_yticklabels(), size=12, fontweight="bold")
ax[1][0].set_xticklabels(ax[1][0].get_xticklabels(), size=12, fontweight="bold")
ax[1][0].set_ylabel(r"Projected-Gradient Energy $\bf{\eta_k^{sub}}$", size=14, fontweight="bold")

ax[1][1].scatter(acc2[0],acc2[1],s = 40 , marker = 'o', facecolors="none",    edgecolors="green")
ax[1][1].scatter(rej2[0],rej2[1],s =40, marker = 'x',   color='red')
ax[1][1].set_ylim(-0.05,1.05)
ax[1][1].set_yticklabels(ax[1][1].get_yticklabels(), size=12, fontweight="bold")
ax[1][1].set_xticklabels(ax[1][1].get_xticklabels(), size=12, fontweight="bold")

ax[1][2].scatter(acc3[0],acc3[1],s = 40 , marker = 'o', facecolors="none",    edgecolors="green")
ax[1][2].scatter(rej3[0],rej3[1],s =40, marker = 'x',   color='red')
ax[1][2].set_ylim(-0.05,1.05)
ax[1][2].set_yticklabels(ax[1][2].get_yticklabels(), size=12, fontweight="bold")
ax[1][2].set_xticklabels(ax[1][2].get_xticklabels(), size=12, fontweight="bold")

ax[1][0].tick_params(length=6, width=3)
ax[1][1].tick_params(length=6, width=3)
ax[1][2].tick_params(length=6, width=3)
ax[1][0].set_xlabel("Iteration", size=14, fontweight="bold")
ax[1][1].set_xlabel("Iteration", size=14, fontweight="bold")
ax[1][2].set_xlabel("Iteration", size=14, fontweight="bold")

plt.savefig(
    "Dimension.svg",
    dpi=400,
    bbox_inches="tight"
)

plt.show()

# Influence of Candidate Basis Sources in Subspace Construction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, NullLocator


def scientific_tick(value, position):
    """
    Format logarithmic ticks as:
    1E-02, 1E-01, 1E+00, 1E+01, ...
    """
    if value <= 0:
        return ""

    exponent = int(np.round(np.log10(value)))
    return f"1E{exponent:+03d}"


def plot_training_validation(
    classical_training,
    classical_validation,
    krylov_training,
    krylov_validation,
    hslm_training,
    hslm_validation,
    sigma_squared=0.65,
    save_name="training_validation_comparison.pdf",
):
    """
    Create a 2-by-3 figure:

        Classical LM       Krylov Subspace LM       HSLM
        Training           Training                  Training
        Validation         Validation                Validation

    Each input should be a list containing one array for each initial guess.
    """

    training_data = [
        classical_training,
        krylov_training,
        hslm_training,
    ]

    validation_data = [
        classical_validation,
        krylov_validation,
        hslm_validation,
    ]

    method_titles = [
        "Randomized",
        "Randomized & Gradient",
        "HSLM",
    ]

    panel_labels = [
        ["A1", "B1", "C1"],
        ["A2", "B2", "C2"],
    ]

    fig, axes = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(15, 6.4),
        sharex=False,
        sharey=False,
    )

    # ---------------------------------------------------------
    # Main formatting parameters
    # ---------------------------------------------------------
    tick_fontsize = 13
    axis_label_fontsize = 14
    title_fontsize = 19
    panel_fontsize = 17.5
    legend_fontsize = 9.4
    line_width = 1.8
    spine_width = 1.3
    tick_width = 1.4
    tick_length = 5

    for column in range(3):

        # =====================================================
        # Top row: training error
        # =====================================================
        ax_train = axes[0, column]

        for run_number, error_values in enumerate(training_data[column], start=1):
            error_values = np.asarray(error_values, dtype=float)
            iterations = np.arange(len(error_values))

            ax_train.plot(
                iterations,
                error_values,
                linewidth=line_width,
                label=f"Initial Guess {run_number}",
            )

        ax_train.axhline(
            y=sigma_squared,
            linestyle="--",
            linewidth=1.7,
            color="gray",
            label=r"$\sigma^2$",
        )

        ax_train.set_title(
            method_titles[column],
            fontsize=title_fontsize,
            fontweight="bold",
            pad=31,
        )

        # =====================================================
        # Bottom row: validation error
        # =====================================================
        ax_validation = axes[1, column]

        for run_number, error_values in enumerate(
            validation_data[column], start=1
        ):
            error_values = np.asarray(error_values, dtype=float)
            iterations = np.arange(len(error_values))

            ax_validation.plot(
                iterations,
                error_values,
                linewidth=line_width,
            )

        # =====================================================
        # Format both rows
        # =====================================================
        for row, ax in enumerate([ax_train, ax_validation]):

            ax.set_yscale("log")
            ax.set_xlim(-0.4, 35.4)
            ax.set_xticks(np.arange(0, 36, 5))

            if row == 0:
                ax.set_ylim(1e-1, 1.5e3)
                ax.set_yticks([1e-1, 1e0, 1e1, 1e2, 1e3])
            else:
                ax.set_ylim(7e-3, 1.5e3)
                ax.set_yticks([1e-2, 1e-1, 1e0, 1e1, 1e2, 1e3])

            ax.yaxis.set_major_formatter(
                FuncFormatter(scientific_tick)
            )

            # Remove all minor ticks
            ax.xaxis.set_minor_locator(NullLocator())
            ax.yaxis.set_minor_locator(NullLocator())

            # Only bottom x ticks and left y ticks
            ax.tick_params(
                axis="x",
                which="major",
                bottom=True,
                top=False,
                labelbottom=True,
                direction="out",
                length=tick_length,
                width=tick_width,
                labelsize=tick_fontsize,
                pad=3,
            )

            ax.tick_params(
                axis="y",
                which="major",
                left=True,
                right=False,
                labelleft=True,
                direction="out",
                length=tick_length,
                width=tick_width,
                labelsize=tick_fontsize,
                pad=3,
            )

            # Make all tick numbers bold
            for tick_label in ax.get_xticklabels():
                tick_label.set_fontweight("bold")

            for tick_label in ax.get_yticklabels():
                tick_label.set_fontweight("bold")

            # Axis-border formatting
            for spine in ax.spines.values():
                spine.set_linewidth(spine_width)
                spine.set_color("0.25")

            # Panel labels: A1, A2, B1, ...
            ax.text(
                -0.01,
                1.045,
                panel_labels[row][column],
                transform=ax.transAxes,
                fontsize=panel_fontsize,
                fontweight="bold",
                horizontalalignment="left",
                verticalalignment="bottom",
            )

    # ---------------------------------------------------------
    # Axis labels
    # ---------------------------------------------------------
    axes[0, 0].set_ylabel(
        "Training Error",
        fontsize=axis_label_fontsize,
        fontweight="bold",
    )

    axes[1, 0].set_ylabel(
        "Validation Error",
        fontsize=axis_label_fontsize,
        fontweight="bold",
    )

    for column in range(3):
        axes[1, column].set_xlabel(
            "Iteration",
            fontsize=axis_label_fontsize,
            fontweight="bold",
        )

    # Legend only in the first panel
    axes[0, 0].legend(
        loc="upper right",
        fontsize=legend_fontsize,
        frameon=False,
        handlelength=2.2,
    )

    plt.subplots_adjust(
        left=0.075,
        right=0.985,
        bottom=0.12,
        top=0.86,
        wspace=0.24,
        hspace=0.32,
    )

    if save_name is not None:
        plt.savefig(
            save_name,
            dpi=400,
            bbox_inches="tight",
        )

    plt.show()



# ============================================================
# Example data
# Replace these arrays with your actual training-error values
# ============================================================

classical_errors = [[512.8412870154503,463.72389882997805, 392.62589482378445, 314.3409036352482, 257.0123251597514,
        209.87241626033355, 149.06728121146185, 106.2873171627373, 69.0641344282651,
        39.840830952422365, 24.31516867102962, 16.484347043457067, 10.651667229458587,
        7.375771063090405, 6.822608625319086, 5.425958577004076, 5.113153075210315,
        4.458826337652293, 3.991927825652915, 3.6757948594151078, 3.4135853857182576,
        3.171879403125056, 2.9684841824561663, 2.8144597378792264, 2.6969132386175354,
        2.6028800820651283, 2.507072338437774, 2.3934495682204915, 2.306090821997932,
        2.2355034840758656, 2.171183946694326, 2.115372902582422, 2.0684442546753186,
        2.027087726452729, 1.990029444031616, 1.955538829986679],
                   [708.5901186584207 ,625.6956234638384, 505.3880323037105, 373.18894680474085, 279.0408964241825,
        232.75427880603272, 164.90944081581597, 94.34801660782692, 56.64985156152514,
        37.02166417875034, 24.595391904858264, 16.163212617450398, 12.089331441804257,
        9.893055624455673, 7.257405705932501, 5.711083042287752, 4.901424717749893,
        4.4109725242599644, 4.052383046274511, 3.7838952435476187, 3.598131911793894,
        3.4440383949499473, 3.2976461954117737, 3.1589704543053383, 3.0350402622471773,
        2.9246386967438847, 2.822874550152949, 2.7238241315263894, 2.6302155866281742,
        2.54043911877099, 2.4427264935183457, 2.3409918742879103, 2.2504223941185266,
        2.1708164048384644, 2.1019100506052477, 2.0387375283490705],
                   [665.1195138503175,589.9678245475901, 480.64276283847175, 359.1745479724556, 265.52226972717773,
        174.7232770671196, 115.54235128894034, 77.07541541611776, 53.42693154833894,
        39.43287148186939, 30.90136347530502, 22.29450472563054, 15.068654401164292,
        11.792996354210867, 7.978684337671142, 5.872930614179996, 4.679569840666483,
        4.430985322273917, 3.8473789491910755, 3.6625118561953407, 3.2962555850377844,
        3.1853731915312595, 2.9277041765340845, 2.714497682428665, 2.524827113271358,
        2.4111593437175225, 2.324159798503868, 2.2540068099761355, 2.183128785811586,
        2.101367384403838, 2.010086571080215, 1.9209840659334663, 1.854193652910684,
        1.8078686426078685, 1.7701668257245609, 1.7329427487030646],
                    [542.8829033663233,489.0098501451666, 410.9229755729275, 324.92659440951905, 257.20256387049164,
        208.29610882786665, 144.91910457731802, 95.03429980800321, 63.256098136676385,
        39.9373697686851, 28.01611091295099, 18.309888239263106, 11.999471464724289,
        10.062899459156903, 7.518959406017291, 6.027664246258343, 5.188056072891021,
        4.665873100031706, 4.21172430848807, 3.9053562227467475, 3.693700238590791,
        3.501009450869734, 3.3285504154940324, 3.180722197688972, 3.0621709195718387,
        2.9589406924174755, 2.8647685020294986, 2.772683006760927, 2.677046159704193,
        2.582575484327661, 2.4858781943181647, 2.4043570251780024, 2.3372796757994863,
        2.2837167167448573, 2.242302645248281, 2.2104912652907673],
                    [602.5307464163853,538.2246468078209, 445.1421165910702, 342.5802317724, 270.10982977870646,
        214.44194121956892, 158.41397562297377, 106.1441569010669, 67.01417274939031,
        41.873328466008104, 26.315611459687656, 18.68452072338596, 13.387355293180729,
        10.183619509169631, 8.18728537512455, 6.9082320490542015, 6.113799082583037,
        5.511959832906495, 5.032568913693871, 4.65222871198829, 4.370384866485137,
        4.169863889323342, 4.0089137787563, 3.855281740812081, 3.707434513212491,
        3.5650774157095455, 3.430302984776083, 3.314286713724606, 3.213605422455989,
        3.1084235640917375, 2.99829292131114, 2.8934173998926402, 2.7946115632714075,
        2.707398811738297, 2.627838484641231, 2.548699426260035]]

krylov_errors = [[512.8412870154503,463.72389882997805, 392.62589482378445, 314.3409036352482, 257.0123251597514,
        209.87241626033355, 138.74579384558353, 66.74725024869892, 49.19162634805159,
        24.204003343731586, 13.593460349157091, 8.589793541241795, 7.76225215826436,
        5.84040668161473, 4.5858548537302894, 3.804100876826803, 3.277398640178889,
        2.8841338294990235, 2.8564267227371163, 2.583200422059895, 2.3482413092242447,
        2.1626230684260825, 1.9977789571871698, 1.8603101978940721, 1.731116510475746,
        1.6280246384962112, 1.537187143341641, 1.4659501759026254, 1.4010529576326958,
        1.3474760781701673, 1.2964062253830966, 1.2524036722380847, 1.2092296206624753,
        1.1710078217834394, 1.133497997093754, 1.100363427992125],
                   [708.5901186584207 ,625.6956234638384, 505.3880323037105, 373.18894680474085, 279.0408964241825,
        232.75427880603272, 166.05296953321263, 64.49459603212298, 32.654438979050916,
        16.557553502653423, 9.927662131083357, 6.43859938022101, 4.84136337425309,
        3.857464644940939, 3.300527197584917, 2.9182775920864947, 2.637731958720956,
        2.4266122904404037, 2.260543091293563, 2.115670858527829, 1.988094244105832,
        1.8726977022447449, 1.7699316098772764, 1.6767129736920288, 1.5960255625802358,
        1.5223277481646211, 1.4602094446173257, 1.4034854906012129, 1.3550994357331707,
        1.3101772845254052, 1.2703849027907759, 1.2324177422859746, 1.198167409549163,
        1.165088248064912, 1.1354026858199409, 1.1061800539281157],
                   [665.1195138503175,589.9678245475901, 480.64276283847175, 359.1745479724556, 265.52226972717773,
        174.7232770671196, 128.19439624044352, 59.25064452276584, 41.33995399631212,
        18.722160368223324, 12.394205443414144, 9.057393921662861, 5.851870272276453,
        4.057360266658441, 3.3177114870526725, 2.8909847562597197, 2.607990476439305,
        2.385926131669804, 2.2082231807719634, 2.0588483152430803, 1.9311571692754943,
        1.8186701679584778, 1.7164934932974014, 1.6266802006808634, 1.5401013034715787,
        1.4635796818520888, 1.390304137426854, 1.3146433195900662, 1.2519158892737021,
        1.1758917120761623, 1.1162795854526102, 1.058124653052957, 1.0137184152708192,
        0.9782090765348761, 0.9502326483789307, 0.9276655381072213],
                    [542.8829033663233, 489.0098501451666, 410.9229755729275, 324.92659440951905, 257.20256387049164,
        208.29610882786665, 129.76451319881355, 60.480510034212585, 33.662715229972534,
        19.38881285848901, 14.681545584298286, 10.75338429989944, 7.995448741902501,
        5.645601843096732, 4.602316655447663, 4.476332843407864, 3.8480262998276142,
        3.4151623656688126, 3.0539884715629886, 2.791641043389835, 2.5330805878963845,
        2.3272142711708153, 2.1313881181981276, 1.9741393252671922, 1.8475044432331578,
        1.741866242489256, 1.6543594004341946, 1.5794361078149501, 1.5151825496401674,
        1.4583282753558426, 1.4071348024831154, 1.36147333147831, 1.318455610903781,
        1.2803964123784106, 1.2437190457835798, 1.21126726427723],
                    [602.5307464163853, 538.2246468078209, 445.1421165910702, 342.5802317724, 270.10982977870646,
        214.44194121956892, 173.16543691785344, 94.69914677059856, 32.54412922967444,
        17.092767812617126, 10.066782434495478, 6.531991830323873, 4.642632049236933,
        3.809202129863415, 3.208241156263578, 2.816845135862132, 2.482322242000651,
        2.247620209695847, 2.0495588712287396, 1.8969821935923021, 1.7589935040536624,
        1.6477296786987048, 1.5501150602125553, 1.4695769050530594, 1.3978898971868654,
        1.338913003306323, 1.2826060066312275, 1.236597869205794, 1.1898129393157533,
        1.1514200931670158, 1.1126287802696844, 1.0803776276690358, 1.0452790711264308,
        1.0167224929626197, 0.9842337709310807, 0.9582362840157413]]
hslm_errors = [[512.8412870154503,463.72389882997805, 392.62589482378445, 314.3409036352482, 257.0123251597514,
        209.87241626033355, 209.6291673687112, 204.05673707841987, 168.9353191390266,
        139.72702477744596, 86.23085580168144, 44.768984033946516, 28.963294953931136,
        23.95189087329867, 11.500777670311425, 10.508231982926157, 6.414509282725617,
        5.727276759971507, 4.235619303355708, 4.010005620066603, 2.910564325977765,
        2.3794907914204435, 2.0896703724538574, 1.6364546257287265, 1.536159158987914,
        1.2548250987692524, 1.136128381921714, 1.0220097112772621, 0.9464042735527586,
        0.8960793265935412, 0.8187416167315461, 0.7691412097603822, 0.7489406106174172,
        0.7162699200254158, 0.7104210954296034, 0.6907711567415281],
                   [708.5901186584207 ,625.6956234638384, 505.3880323037105, 373.18894680474085, 279.0408964241825,
        232.75427880603272, 229.76366262581362, 229.76366262581362, 229.76366262581362,
        225.76144475089347, 213.74653336533848, 194.6628442757715, 168.25224932618514,
        127.54095078617223, 53.671906357855605, 44.74529845091877, 34.7859649454097,
        12.786081247639673, 10.73357211748653, 8.884999558239057, 4.731116145762712,
        3.704564241119653, 2.7966926486105996, 2.2560422292930347, 1.623498158858321,
        1.3368037303283393, 1.137907541452071, 1.0772648820939084, 0.7994429661358133,
        0.7758715755182544, 0.7086238976285838, 0.6945710356343309, 0.6714796716999699,
        0.6598188538000735, 0.6593274341372264, 0.6440983193609469],
                   [665.1195138503175,589.9678245475901, 480.64276283847175, 359.1745479724556, 265.52226972717773,
        174.7232770671196, 164.90502405277124, 154.01107471977778, 135.88850481467765,
        72.16591841358726, 66.31636752142171, 55.085542277714794, 39.07145869111133,
        37.11331761658976, 19.572825861682613, 17.72781484742437, 11.449457012150852,
        8.893240609224506, 7.422358159049032, 6.126018426770483, 4.855054102338361,
        3.7290382894483876, 3.09861111598704, 2.2850346880380465, 2.097430547040545,
        1.1593942695223236, 0.9223623486481287, 0.8313714210394703, 0.7419151054848262,
        0.7115479175339136, 0.6810673268535123, 0.6660679626093794, 0.6525087542928164,
        0.6445049246186654, 0.6399607343382244, 0.6336148419116318],
                    [542.8829033663233, 489.0098501451666, 410.9229755729275, 324.92659440951905, 257.20256387049164,
        208.29610882786665, 203.93886582572307, 197.4424944428288, 179.67319432334116,
        122.75928126269933, 100.56083753280771, 62.09178580355202, 62.01443280323106,
        27.51266004081089, 19.987114375193645, 19.089057368788566, 16.93566235471257,
        5.182024587264339, 4.232513912099861, 3.462138014549774, 2.0932519448932636,
        1.7064347482911602, 1.4726149420377506, 1.2311203137211246, 1.1507447240192101,
        0.9813844178320568, 0.8840911353965636, 0.8352048169745685, 0.7730615866532727,
        0.7099557792283091, 0.6697724137214834, 0.661472940330404, 0.6470411854783702,
        0.6414747455253536, 0.6359974769534767, 0.6333536212716901],
                    [602.5307464163853,538.2246468078209, 445.1421165910702, 342.5802317724, 270.10982977870646,
        214.44194121956892, 209.63704311583624, 205.53436877110195, 189.4129676999117,
        121.35721978669227, 96.59402579262904, 81.78221386473763, 51.29648419438728,
        45.06660946336463, 39.4533896157925, 15.813685688529455, 14.281810088443384,
        7.594474602668143, 6.484427398802411, 6.096901856681865, 3.93136076014852,
        3.7332115178956817, 2.7772463816503845, 2.576515639852803, 2.0092854408109684,
        1.5542527092767244, 1.4396611485412578, 1.2160032561784433, 1.1308523308774108,
        1.02625979196832, 0.8282007813214088, 0.7624220629900609, 0.7152418779170502,
        0.6760030064487506, 0.6711607082896714, 0.6529549502423045]]

classical_validation_errors = [[542.4040857464349,489.8502601141796, 413.37933200693215, 327.8327215216789, 263.73627582020686,
        215.1578254813226, 149.90742099511675, 105.79239406193793, 68.15981442002159,
        39.212828176493936, 23.428840488473277, 15.480038664326857, 9.599911933686535,
        6.8640536063333295, 6.384386504062151, 4.907559002577287, 4.586453316637626,
        3.9962751111991137, 3.495730649579211, 3.1817924918650915, 2.922368955328576,
        2.6073233315605, 2.3272472736449945, 2.1455645812646105, 2.0484962546615226,
        1.9919781697341885, 1.904070389716837, 1.7730633045633124, 1.7021772946972114,
        1.6454779001628257, 1.5860816770884463, 1.5259688506127038, 1.473623849252802,
        1.4312707647531902, 1.3941386887518352, 1.3566199687364424],
                   [743.220635032485,657.2695793245807, 532.3023804655686, 393.85045198389275, 291.83985348514625,
        238.01485940903015, 159.6412987586928, 89.97650471302359, 53.05871099703959,
        32.92374040732569, 22.181801263199777, 14.856709447036694, 10.606615890260194,
        8.545325099050725, 5.994070620779419, 4.913237409286248, 4.283443170709725,
        3.8004102922025926, 3.401580833395266, 3.09545140091555, 2.8788979840491944,
        2.7058430225817887, 2.5519270903309934, 2.418152585555542, 2.3039344553426613,
        2.2060661194816458, 2.1180598471057737, 2.040624443989905, 1.9836042627469666,
        1.9256767771191239, 1.8557252742062, 1.762926826689525, 1.6499538825348663,
        1.569089267296285, 1.514815775070405, 1.4645647483751645],
                   [710.4990215410855,629.1674834150868, 509.8268749326498, 376.5170640773331, 274.2108582814963,
        180.15090151734978, 114.57612210730986, 74.84988073449216, 52.42274280367291,
        39.30875679322634, 30.378914543873446, 21.020361458610083, 14.477070318326486,
        11.222195968314205, 7.115880816800092, 4.8852559682566215, 4.005483952349538,
        3.7782997900236523, 3.4085120912319007, 3.273103328628031, 2.8904312331530213,
        2.84334717376169, 2.5118351343097056, 2.293378700602283, 2.071589259109093,
        1.9015512540523742, 1.7584430028625764, 1.64756521630429, 1.5591399359993559,
        1.4652728401965274, 1.3740552228130813, 1.275400839560107, 1.2070348711615075,
        1.1654987581959018, 1.1426267199617082, 1.1204184443625467],
                    [580.7913640865677,521.8964146872299, 436.2216453862869, 341.1326269834507, 265.5118766645761,
        209.17703630140477, 143.31035743330736, 92.39150210669101, 61.152696056050786,
        38.39472081300541, 26.892791642722756, 17.479834398542447, 10.789473975406406,
        8.693790310309025, 6.373030642923677, 5.027735647700989, 4.324520103163959,
        3.8196300051366343, 3.382220833268277, 3.155993833007314, 3.0535349883452128,
        2.9874588150255494, 2.869328724077942, 2.712338763193547, 2.559576766967616,
        2.4122713515576004, 2.2773786069768853, 2.1553263295090734, 2.043137504568672,
        1.946806029535226, 1.8623593443824458, 1.7951469097581088, 1.7395252124257456,
        1.7153509851639028, 1.7121902068924615, 1.711218692852162],
                    [642.2234912964643,573.3402858917676, 472.95479849182163, 361.18570575686454, 281.03256433982796,
        217.15946741246904, 161.0716944803627, 105.7702059846986, 67.84608935266434,
        41.64345916878531, 25.831953420116758, 18.157786439076663, 12.951569614778432,
        9.328267387830284, 7.080252009196131, 5.867623610598767, 5.219793886847395,
        4.773677468527186, 4.43853389060404, 4.140084264078161, 3.885020972958095,
        3.6916872258533266, 3.5403459586055286, 3.392422068307136, 3.2409084251882887,
        3.082237708296285, 2.93050188815217, 2.7974676763116664, 2.6749505368645723,
        2.534938748418131, 2.3828071110106865, 2.260300241578084, 2.1243390679483345,
        1.9974269909461073, 1.8960324292021662, 1.8088496849857543]]
krylov_validation_errors = [[542.4040857464349,489.8502601141796, 413.37933200693215, 327.8327215216789, 263.73627582020686,
        215.1578254813226, 135.45386427507742, 62.4886561952628, 46.82318971830368,
        22.431325476584064, 12.23338930230302, 7.6953095067899016, 6.846505118991662,
        4.695355925328903, 3.80104849443132, 3.150484848749867, 2.74275107192786,
        2.377492426126361, 2.332495492232627, 2.0685788957955413, 1.8470221683071455,
        1.6287911914467574, 1.4617594517479113, 1.3045968229193214, 1.149791903590501,
        1.0411580745062492, 0.9263213982674826, 0.8558138522853987, 0.7792204266806998,
        0.7265989770837143, 0.6718373459482533, 0.626545631841766, 0.5864564166669558,
        0.5445616537954732, 0.5146947821558461, 0.4771454130266933],
                   [743.220635032485,657.2695793245807, 532.3023804655686, 393.85045198389275, 291.83985348514625,
        238.01485940903015, 163.12515396365157, 59.85407361035168, 28.771241917597415,
        15.36718758694927, 8.772375152526834, 5.429902387254775, 4.126071685789657,
        3.222975388025664, 2.693981027641573, 2.4201261063766233, 2.0494779883302456,
        1.9252751054880637, 1.6807805286629196, 1.5618636092876619, 1.4092161353977586,
        1.2909968942517693, 1.1916251459133733, 1.0867051040899707, 1.0195998515543374,
        0.9341589194713565, 0.8865276056753439, 0.8219133867997326, 0.7810673315041327,
        0.7339597624602828, 0.6937006806446598, 0.6577557581600826, 0.6194585362712128,
        0.5882265117753513, 0.5555073102018351, 0.5247006762074459],
                   [710.4990215410855,629.1674834150868, 509.8268749326498, 376.5170640773331, 274.2108582814963,
        180.15090151734978, 124.2485749468178, 56.92640808129082, 40.85255608838282,
        17.227229040987247, 11.609302898437546, 8.150360634941748, 5.281378293072186,
        3.4630230547856744, 2.739113925637019, 2.2095238791711433, 1.9588509698756602,
        1.757433038492698, 1.5714223753107612, 1.4415237697696444, 1.3094523976454484,
        1.1756545114699193, 1.081373316218198, 0.9499633577046442, 0.8933733127572373,
        0.7926268576549556, 0.7411448945150836, 0.6649594276101685, 0.5995292138872802,
        0.5326019707176183, 0.47025384433538736, 0.42088082686501416, 0.3686454350690461,
        0.3458928480674746, 0.3090408960919501, 0.2967054341698038],
                    [580.7913640865677, 521.8964146872299, 436.2216453862869, 341.1326269834507, 265.5118766645761,
        209.17703630140477, 125.04672488029843, 58.81698804204181, 31.731028535916476,
        17.78767713134138, 13.806818107250253, 9.578885101072629, 6.66490903519773,
        4.45095488873577, 3.458800390065122, 3.370367608860407, 2.9956567556418423,
        2.571137511737606, 2.356759829116457, 2.0719947251466913, 1.9161078103072156,
        1.6659903633630457, 1.5390347011395673, 1.3436620961935777, 1.2332737713251654,
        1.1290284972254439, 1.0322526677230393, 0.9654875754380804, 0.8906415643589487,
        0.843178750184029, 0.7810481889631139, 0.7477647717397468, 0.6908414337483176,
        0.6691088992407271, 0.6164099780671111, 0.60192033525999],
                    [642.2234912964643,573.3402858917676, 472.95479849182163, 361.18570575686454, 281.03256433982796,
        217.15946741246904, 167.02129484758743, 93.69081454430894, 31.13653049309815,
        15.824899054723064, 8.922491339698167, 5.656017674451806, 3.903425833837772,
        2.9163990150674217, 2.3619843378829795, 1.9767103162953392, 1.6348195678502992,
        1.4657546910797046, 1.2698106814547063, 1.151024140396545, 1.0226333699714083,
        0.9304201416101828, 0.8457666529290807, 0.7743517786379365, 0.715932652154494,
        0.6646163035654468, 0.615521256745747, 0.5813861349300748, 0.5326119632871226,
        0.5095817735086614, 0.468379104242394, 0.4480287872529591, 0.41468687771256996,
        0.3933350445686203, 0.3616783902757038, 0.341729031989346]]
hslm_validation_errors = [[542.4040857464349,489.8502601141796, 413.37933200693215, 327.8327215216789, 263.73627582020686,
        215.1578254813226, 214.9101161694597, 210.20393632619357, 170.67406291919917,
        141.11946532407293, 85.6205646538486, 41.84315824937307, 27.511209686781225,
        22.822845569005455, 10.054900115877444, 9.456197373167733, 5.188237714803337,
        4.86048932446515, 3.3812454114425434, 3.397065555144495, 2.0969096121628645,
        1.7096986482575551, 1.3709676222916782, 0.9844485583585569, 0.8605556053390958,
        0.6185570652564385, 0.5275100955884734, 0.37528512927350677, 0.3311266503791668,
        0.26051368449944884, 0.19851108173853513, 0.15794966831271112, 0.13243980833685776,
        0.10813397915737767, 0.10033513519096414, 0.08693598829073736],
                   [743.220635032485, 657.2695793245807, 532.3023804655686, 393.85045198389275, 291.83985348514625,
        238.01485940903015, 235.12571681748344, 235.12571681748344, 235.12571681748344,
        230.72124369347566, 219.4648509991563, 201.250262122966, 177.37252650152368,
        134.0027316918896, 53.822737422415244, 44.76616979671488, 35.005564727625426,
        11.990287696360884, 9.819085436441508, 8.07401764174454, 4.069234186011928,
        2.9716685198855792, 2.1392024397604366, 1.6684043246908633, 0.9665420232385844,
        0.7274854525663209, 0.5346663539856478, 0.4543760804085982, 0.18421052553134973,
        0.16863371630570084, 0.10051912579764088, 0.10007148347399698, 0.0701880906623985,
        0.061463147695335306, 0.06507001530467998, 0.047891584679933104],
                   [710.4990215410855,629.1674834150868, 509.8268749326498, 376.5170640773331, 274.2108582814963,
        180.15090151734978, 168.29418558118317, 156.47823896155396, 138.13778983442313,
        73.7099140108757, 67.2894904020078, 56.00863097416815, 39.41706200079122,
        36.46025095739519, 18.634809957079348, 17.22131326038768, 10.75893648154374,
        8.27413754200488, 6.5835620104462125, 5.5815596818582005, 4.0326983723333685,
        3.0250211686411257, 2.372186459263582, 1.6256282446353727, 1.4976795399977028,
        0.5315081769006939, 0.32986808184915856, 0.22486786984631632, 0.15001354651450718,
        0.11488830771389562, 0.08973716664886604, 0.07087690633228752, 0.05921619933711017,
        0.04979198427921048, 0.047182010266027, 0.03957095871275692],
                    [580.7913640865677,521.8964146872299, 436.2216453862869, 341.1326269834507, 265.5118766645761,
        209.17703630140477, 204.26291106897781, 197.36849619878322, 181.67321018076757,
        121.65207252838506, 99.5442352048404, 60.387773642309725, 62.17745009327641,
        26.49825436650731, 18.94910502032432, 18.658412189799304, 16.644367678878645,
        4.138468798132913, 3.358123639132337, 2.6714864628350012, 1.3356781858810378,
        1.0026850041000177, 0.7995460137064369, 0.5681076923653099, 0.4890933611278769,
        0.3278695105112319, 0.2393908003845374, 0.19451411512579736, 0.14754181479463416,
        0.08873832401682961, 0.05706187016994341, 0.051534342018888166, 0.044905052987449935,
        0.037690833464851387, 0.03551254846566317, 0.03566129445652601],
                    [642.2234912964643,573.3402858917676, 472.95479849182163, 361.18570575686454, 281.03256433982796,
        217.15946741246904, 211.60128261787528, 206.6209875160558, 192.1320836295117,
        121.22417311249652, 96.00697366826434, 81.34622929408928, 50.039274772635046,
        43.756075509787834, 38.861219937855246, 13.859664620811428, 12.952804516883148,
        6.1273329737392475, 5.151884087426171, 4.940101967926901, 3.048822403902517,
        2.9473228305086394, 1.9910060877126048, 1.877812211717921, 1.2932144444325044,
        0.8939943492453823, 0.7621581657813182, 0.5796870922408368, 0.4846763472207142,
        0.41255581697877214, 0.21259789058515327, 0.14644414468949163, 0.1096602239381408,
        0.07125242148573491, 0.06831342362135892, 0.05166257447373651]]
plot_training_validation(
    classical_training=classical_errors,
    classical_validation=classical_validation_errors,
    krylov_training=krylov_errors,
    krylov_validation=krylov_validation_errors,
    hslm_training=hslm_errors,
    hslm_validation=hslm_validation_errors,
    sigma_squared=0.622,
    save_name="training_validation_comparison4.svg",
)
